In [1]:
import rebound 
import numpy as np
import matplotlib.pyplot as plt
import math
import seaborn as sns
from astropy import units as u


Our orbital evolution simulations were undertaken using the REBOUND package which can integrate the motion of particles under the influence of gravity. In this way, the gravitational interaction among star and planets are taken into account in our simulation.

In addition to the star-planet, planet-planet interaction, the disk-planet interaction also play a key role in the dynamical processes. 

We accept the disk model introduced in \cite{2017ApJ...836..207Z} and characterize the contribution of the gravity from the disk for planet b and planet c separately, as planet b is massive enough to open a gap which will influence the apsidal precession of the planet. 

The thin-disk model with an axisymmetric surface density distribution, $\Sigma(r, t)$ at the radius $r$ and time $t$, id of the form

\begin{equation}
\Sigma(r,t)=\Sigma(r_0,0)f_{dep}(t)(r/r_0)^{-k}
\end{equation}

where $r_0 =1 au$ and $f_{dep}(t)=exp(-t/T_{dep})$ is the time-dependent depletion function of the gas disk. In our model(ref needed), k=1.01 with an initial surface density $ \Sigma{ (r_0, 0)} = 50 g\cdot cm^{-2}$ at the location of $r = 1 au$.

For planet b, the force from disk potential should be \citep{nagasawaida2000,ward1981}:
\begin{equation}
F(r,t) =  2  \pi G \Sigma(r,t) \sum_{n=0}^{\infty} A_{n}\{ 
		( \frac{2n}         {2n-1+k}       )
		( \frac{r}          {R_{\rm out}}  )^{2n-1+k} \nonumber \\
	   - ( \frac{2n+1}       {2n+2-k}      )
  	    ( \frac{R_{\rm in}} {r}            )^{2n+2-k} 	\}.
\end{equation}

where $G$ is the gravitational constant, $A_n$ is a constant.


For planet c, the apsidal precession caused by the gas disk is dominated by the gas in their neighborhood. We also consider the condition where planet b is close to $R_{in}$, so the force expression is:

\begin{equation}
 F  =  - 4\pi G \Sigma(r,t)\{Z_k + \nonumber
	\sum _{n=0}^{\infty} A_{n} 
	(\frac{n}{2n-1+k}) 
	(\frac{r}{R_{\rm in}})^{2n-1+k} \}. 
\end{equation}

In addition, the eccentricity of planet c is also damped and excited through the interaction with the disk  \citep{goldreichtremaine1980, ward1997}. With a simple approximation (Zhou et al. 2007), the force cause the eccentricity damping can be written as:

\begin{equation}
{\bf F}_{\rm D} = - \frac{{\bf V} - {\bf V}_{\rm kep}^{\prime}}
{T_{\rm damp}/f_e} \quad .
\end{equation}

where $f_e$ is the coefficient and we set it as 1. We adopted the timescale of Type 1 migration to characterize the eccentricity damping  mechanism \citep{artymowicz1993,ward1993}:

\begin{equation} 
T_{\rm damp,t} \simeq 
	\left(\frac{M_{\ast}}{M_{p}}\right)
	\left(\frac{M_{\ast}}{\Sigma a^2} \right)
	\left(\frac{c_s}{a\Omega_K}\right)^4 \Omega_{k}^{-1} 
\end{equation}

where $\Omega_k = (GM_*/a^3)^{1/2}$ is the Kepler frequency, and $c_s = H(r)\cdot \Omega_k$ is the sound speed of the gas disk. $H(r) = H_0\cdot (\frac{r_p}{1au})^{1.25}$ is the disk scale height and we accept $H_0 = 0.05 au$ in this work.

# example case run can be found in rebound+SSR+type1_baseline.py

In [2]:
def diskpotential_w_gap(reb_sim):

    dens0,di,h0,hi,zk,tdep,mcrit,gama,rt,rm,coef,q1,q2,f_e,f_a = define_variables(1)
    #2-D gas disk potential
    G = sim.G;
    t = sim.t;
    particles = sim.particles;
    com = particles[0];

    A=np.zeros(10);
    A[0] = 1.0;
    for j in np.arange(1,10,1):
        A[j] = A[j-1]*(2.*j-1)*(2.*j-1)/4./j/j;
    
    N = sim.N;
    
    if N>1:
        for n in range(n_j):
            p = particles[n+1];

            x = p.x - com.x;
            y = p.y - com.y;
            z = p.z - com.z;
            rp = np.sqrt(x*x + y*y + z*z);

            o = p.calculate_orbit(primary=com)
            dens = dens0 * pow(o.a,-di) * np.exp(-t/tdep);
            ain = o.a*(1-o.e)*(1 - pow(p.m/com.m/3., 0.333));
            aout = o.a*(1+o.e)*(1 + pow(p.m/com.m/3., 0.333));
            if o.a > 0:
                #printf("%e\t%e\n", ain, aout);
                nt_sum = 0.;
                for j in range(10):
                    nt = 2*(j)*A[j]*pow(o.a/aout,2*(j)-1+di)/(2*(j)-1+di) - (2*(j)+1)*A[j]*pow(ain/o.a,2*(j)+2-di)/(2*(j)+2-di);
                    nt_sum += nt;

                f = 2.*M_PI*G*dens*nt_sum;
        #         f = -4.*M_PI*G*dens*zk
            else:
                f = 0.;
            p.ax += f*x/rp;
            p.ay += f*y/rp;
            p.az += f*z/rp;
    
    if N>1+n_j:
        for n in range(sim.N-1-n_j):
            p2 = particles[n+1+n_j];

            x2 = p2.x - com.x;
            y2 = p2.y - com.y;
            z2 = p2.z - com.z;
            rp2 = np.sqrt(x2*x2 + y2*y2 + z2*z2);

            o2 = p2.calculate_orbit(primary=com)
            dens2 = dens0 * pow(o2.a,-di) * np.exp(-t/tdep);
            if o2.a > 0:
                #printf("%e\t%e\n", ain, aout);
                f2 = -4.*M_PI*G*dens2*zk
                p2.ax += f2*x2/rp2;
                p2.ay += f2*y2/rp2;
                p2.az += f2*z2/rp2;
            else:
                f2 = 0
                p2.ax += 0
                p2.ay += 0
                p2.az += 0
#                 sim.remove(n+2)
        

In [3]:

def diskpotential_wo_gap(reb_sim):

    dens0,di,h0,hi,zk,tdep,mcrit,gama,rt,rm,coef,q1,q2,f_e,f_a = define_variables(1)
    #2-D gas disk potential
    G = sim.G;
    t = sim.t;
    particles = sim.particles;
    com = particles[0];


    N = sim.N;
    
    for n in range(sim.N-1):
        p = particles[n+1];

        x = p.x - com.x;
        y = p.y - com.y;
        z = p.z - com.z;
        rp = np.sqrt(x*x + y*y + z*z);

        o = p.calculate_orbit(primary=com)
        dens = dens0 * pow(rp,-di) * np.exp(-t/tdep);
        if o.a > 0:
            f = -4.*M_PI*G*dens*zk
            p.ax += f*x/rp;
            p.ay += f*y/rp;
            p.az += f*z/rp;

        # else:
            # sim.remove(n+1)

    

In [4]:


def typeI(reb_sim):
    dens0,di,h0,hi,zk,tdep,mcrit,gama,rt,rm,coef,q1,q2,f_e,f_a = define_variables(1)
    # Velocity dependent type I damping
    G = sim.G
    t = sim.t
    particles = sim.particles;
    com = particles[0];
    N = sim.N;
    
    if N>1+n_j:
        for n in range(sim.N-n_j-1):
            p =(particles[n+1+n_j]);
            x = p.x - com.x;
            y = p.y - com.y;
            z = p.z - com.z;
            rp = np.sqrt(x*x + y*y + z*z);
            vx = p.vx - com.vx;
            vy = p.vy - com.vy;
            vz = p.vz - com.vz;
            mu = G*(com.m + p.m);
            q = p.m / com.m;

            o = p.calculate_orbit(primary=com)
            if o.a >0:
                dens = dens0 * pow(rp,-di) * np.exp(-t/tdep);
                h = h0 * pow(rp,hi);
                Omega_K=np.sqrt(G*com.m/o.a**3)
                cs = Omega_K * h
                t_damp = pow(p.m,-1)*pow(dens*o.a**2,-1)*pow(cs/(o.a*Omega_K),4)*pow(Omega_K,-1);

                #force of eccentricity damping
                t_e = t_damp / f_e;
                vr = (x*vx + y*vy + z*vz)/rp;
                temp = vr/t_e;
                p.ax -= temp*x/rp;
                p.ay -= temp*y/rp;
                p.az -= temp*z/rp;

            if (open_typeI == 2):
                #force of type I migration
                rtt2 = pow(rp/rt,2);
                f_nsc = coef * (1 - 2*rtt2/(1+rtt2));
                rrm4 = pow(rp/rm,4);
                f_lb = 1 - 2*rrm4/(1+rrm4);      
                qq12 = pow(q/q1,2);
                qq22 = pow(q/q2,-2);
                qq = qq12*qq22/(2*qq12*qq22+qq12+qq22);

                fa = (f_nsc*qq + f_lb);
                t_a = t_damp /f_a/fa/h/h;

                dvx = vx - vr*x/rp;
                dvy = vy - vr*y/rp;
                dvz = vz - vr*z/rp;
                dv = np.sqrt(dvx*dvx + dvy*dvy + dvz*dvz);
                temp = np.sqrt(G * com.m / rp)/ 2. / t_a;
                p.ax += temp*dvx/dv;
                p.ay += temp*dvy/dv;
                p.az += temp*dvz/dv;